# BEAD Random Forest — Final Model
Log-transformed target with 6 features (post-ablation).

Dropped after ablation (zero or negative contribution): `avg_latency`, `priority_broadband_project`, `incumbent_democrat`

Includes SHAP and LIME interpretability analysis.

In [1]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

client = bigquery.Client(project='broadband-data')
print('Connected to BigQuery: broadband-data')

Connected to BigQuery: broadband-data


In [5]:
# Load all data sources
df_projects = client.query("""
SELECT project_id, state, bead_support, estimated_miles_aerial_fiber,
       estimated_miles_buried_fiber, estimated_jobs, project_type, priority_broadband_project
FROM `broadband-data.fp_approved.deployment_projects`
""").to_dataframe()

df_locations = client.query("""
SELECT project_id, COUNT(*) AS funded_locations,
       SAFE_CAST(APPROX_TOP_COUNT(CAST(technology AS STRING), 1)[OFFSET(0)].value AS FLOAT64) AS technology,
       AVG(CAST(low_latency AS INT64)) AS avg_latency
FROM `broadband-data.fp_approved.locations`
GROUP BY project_id
""").to_dataframe()

state_name_to_abbr = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
    'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'Florida': 'FL', 'Georgia': 'GA',
    'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA',
    'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV', 'New_Hampshire': 'NH',
    'New_Jersey': 'NJ', 'New_Mexico': 'NM', 'New_York': 'NY', 'North_Carolina': 'NC',
    'North_Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA',
    'Rhode_Island': 'RI', 'South_Carolina': 'SC', 'South_Dakota': 'SD', 'Tennessee': 'TN',
    'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA',
    'West_Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY', 'District_of_Columbia': 'DC'
}
df_nbm = client.query("""
SELECT state, COUNT(DISTINCT frn) AS state_num_providers
FROM `broadband-data.fcc_bdc.nbm_hive` GROUP BY state
""").to_dataframe()
df_nbm['state'] = df_nbm['state'].map(state_name_to_abbr)
df_nbm = df_nbm.dropna(subset=['state'])

df_state_pop = client.query("""
SELECT stateabbr AS state, SUM(pop2020) AS state_population
FROM `broadband-data.fcc_block_level_pop.us2020` GROUP BY stateabbr
""").to_dataframe()

print(f'Projects: {df_projects.shape}, Locations: {df_locations.shape}')

Projects: (5740, 8), Locations: (5715, 4)


In [6]:
# Merge + feature engineering
df = df_projects.merge(df_locations, on='project_id', how='left')
df = df.merge(df_nbm, on='state', how='left')
df = df.merge(df_state_pop, on='state', how='left')

df['funded_locations'] = df['funded_locations'].fillna(0)
df['technology'] = df['technology'].fillna(0)
df['state_num_providers'] = df['state_num_providers'].fillna(0)
df['state_population'] = df['state_population'].fillna(0)

df['total_fiber_miles'] = df['estimated_miles_aerial_fiber'].fillna(0) + df['estimated_miles_buried_fiber'].fillna(0)
df['miles_per_location'] = df['total_fiber_miles'] / df['funded_locations'].replace(0, np.nan)
df['miles_per_location'] = df['miles_per_location'].fillna(0)
df['jobs_per_location'] = df['estimated_jobs'] / df['funded_locations'].replace(0, np.nan)
df['jobs_per_location'] = df['jobs_per_location'].fillna(0)

df['funding_per_location'] = df['bead_support'] / df['funded_locations'].replace(0, np.nan)
df = df.dropna(subset=['funding_per_location'])
low = df['funding_per_location'].quantile(0.025)
high = df['funding_per_location'].quantile(0.975)
df = df[(df['funding_per_location'] >= low) & (df['funding_per_location'] <= high)]

df['log_funding'] = np.log1p(df['funding_per_location'])
print(f'Samples: {df.shape[0]}')

Samples: 5429


In [7]:
from sklearn.preprocessing import OrdinalEncoder

state_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
state_encoded = state_enc.fit_transform(df[['state']])
df['state_encoded'] = state_encoded

In [13]:
# Final model: 6 features, log target
feature_cols = [
    'miles_per_location',    # Ablation R² drop: +0.039
    'technology',            # Ablation R² drop: +0.039
    'jobs_per_location',     # Ablation R² drop: +0.020
    #'state_population',      # Ablation R² drop: +0.010
    'total_fiber_miles',     # Ablation R² drop: +0.006
    #'state_num_providers',   # Ablation R² drop: +0.001
    'state_encoded'
]

X = df[feature_cols].fillna(0)
y = df['log_funding']
groups = df['state']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

rf = RandomForestRegressor(n_estimators=200, min_samples_split=5, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

r2 = r2_score(y_test, y_pred)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2 = cross_val_score(rf, X, y, cv=kf, scoring='r2')
rmse = np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(y_pred)))
mae = mean_absolute_error(np.expm1(y_test), np.expm1(y_pred))

print('=== Final Model: 6 Features, log(funding_per_location) ===')
print(f'Test R² (log):   {r2:.4f}')
print(f'CV R² (log):     {cv_r2.mean():.4f} +/- {cv_r2.std():.4f}')
print(f'RMSE (real $):   {rmse:,.0f}')
print(f'MAE (real $):    {mae:,.0f}')

=== Final Model: 6 Features, log(funding_per_location) ===
Test R² (log):   0.7637
CV R² (log):     0.7592 +/- 0.0197
RMSE (real $):   3,068
MAE (real $):    1,695


In [20]:
X.shape, y.shape, groups_train.shape

((5429, 6), (5429,), (4793,))

In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, GroupShuffleSplit, GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Final model: 6 features, log target
feature_cols = [
    'miles_per_location',    # Ablation R² drop: +0.039
    'technology',            # Ablation R² drop: +0.039
    'jobs_per_location',     # Ablation R² drop: +0.020
    'state_population',      # Ablation R² drop: +0.010
    'total_fiber_miles',     # Ablation R² drop: +0.006
    'state_num_providers',   # Ablation R² drop: +0.001
]

X = df[feature_cols].fillna(0)
y = df['log_funding']
groups = df['state']

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

rf = RandomForestRegressor(n_estimators=200, min_samples_split=5, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

r2 = r2_score(y_test, y_pred)
kf = GroupKFold(n_splits=5, shuffle=True, random_state=42)
cv_r2 = cross_val_score(
    rf, X, y, cv=kf, groups=groups, scoring='r2', n_jobs=-1,
)
rmse = np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(y_pred)))
mae = mean_absolute_error(np.expm1(y_test), np.expm1(y_pred))

print('=== Final Model: 6 Features, log(funding_per_location) ===')
print(f'Test R² (log):   {r2:.4f}')
print(f'CV R² (log):     {cv_r2.mean():.4f} +/- {cv_r2.std():.4f}')
print(f'RMSE (real $):   {rmse:,.0f}')
print(f'MAE (real $):    {mae:,.0f}')

=== Final Model: 6 Features, log(funding_per_location) ===
Test R² (log):   0.4649
CV R² (log):     0.4433 +/- 0.1303
RMSE (real $):   4,845
MAE (real $):    3,212


In [ ]:
# Predicted vs Actual + Feature Importance
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Log scale
axes[0].scatter(y_test, y_pred, alpha=0.4, s=20, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--')
axes[0].set_xlabel('Actual log(Funding/Location)')
axes[0].set_ylabel('Predicted')
axes[0].set_title(f'Log Scale | R²={r2:.3f}')

# Real $ scale
yt_real = np.expm1(y_test)
yp_real = np.expm1(y_pred)
axes[1].scatter(yt_real, yp_real, alpha=0.4, s=20, color='steelblue')
lims2 = [min(yt_real.min(), yp_real.min()), max(yt_real.max(), yp_real.max())]
axes[1].plot(lims2, lims2, 'r--')
axes[1].set_xlabel('Actual Funding/Location ($)')
axes[1].set_ylabel('Predicted')
axes[1].set_title('Real $ Scale')

# Feature importance
imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values()
imp.plot.barh(ax=axes[2], color='steelblue')
axes[2].set_xlabel('Importance')
axes[2].set_title('Feature Importances')

plt.tight_layout()
plt.show()

## SHAP Analysis

In [ ]:
# Install if needed
try:
    import shap
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'shap'])
    import shap
print(f'SHAP version: {shap.__version__}')

In [ ]:
# Compute SHAP values
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)
print(f'SHAP values shape: {shap_values.shape}')

In [ ]:
# SHAP summary plot — shows direction + magnitude of each feature's effect
shap.summary_plot(shap_values, X_test, feature_names=feature_cols, show=True)

In [ ]:
# SHAP bar plot — mean absolute SHAP value per feature
shap.summary_plot(shap_values, X_test, feature_names=feature_cols, plot_type='bar', show=True)

In [ ]:
# SHAP dependence plots for top 3 features
top3 = pd.Series(np.abs(shap_values).mean(axis=0), index=feature_cols).nlargest(3).index.tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, feat in enumerate(top3):
    shap.dependence_plot(feat, shap_values, X_test, feature_names=feature_cols, ax=axes[i], show=False)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP waterfall for individual predictions
# Show highest and lowest funded projects
idx_high = y_test.idxmax()
idx_low = y_test.idxmin()

pos_high = list(X_test.index).index(idx_high)
pos_low = list(X_test.index).index(idx_low)

print('=== Highest Funded Project ===')
print(f'Actual: ${np.expm1(y_test.loc[idx_high]):,.0f}/location')
print(f'Predicted: ${np.expm1(y_pred[pos_high]):,.0f}/location')
shap.plots.waterfall(shap.Explanation(values=shap_values[pos_high], 
                                       base_values=explainer.expected_value,
                                       data=X_test.iloc[pos_high],
                                       feature_names=feature_cols), show=True)

print('\n=== Lowest Funded Project ===')
print(f'Actual: ${np.expm1(y_test.loc[idx_low]):,.0f}/location')
print(f'Predicted: ${np.expm1(y_pred[pos_low]):,.0f}/location')
shap.plots.waterfall(shap.Explanation(values=shap_values[pos_low],
                                       base_values=explainer.expected_value,
                                       data=X_test.iloc[pos_low],
                                       feature_names=feature_cols), show=True)

## LIME Analysis

In [ ]:
try:
    import lime
    import lime.lime_tabular
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'lime'])
    import lime
    import lime.lime_tabular
print('LIME imported successfully')

In [ ]:
# Create LIME explainer
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=feature_cols,
    mode='regression',
    random_state=42
)
print('LIME explainer ready')

In [ ]:
# LIME explanations for 3 representative projects
# High, median, and low funding predictions
pred_series = pd.Series(y_pred, index=range(len(y_pred)))
idx_high = int(pred_series.idxmax())
idx_med = int(pred_series.sub(pred_series.median()).abs().idxmin())
idx_low = int(pred_series.idxmin())

y_test_arr = y_test.values

for label, pos in [('High Funding', idx_high), ('Median Funding', idx_med), ('Low Funding', idx_low)]:
    actual = np.expm1(y_test_arr[pos])
    predicted = np.expm1(y_pred[pos])
    
    print(f'\n=== {label} ===')
    print(f'Actual: ${actual:,.0f}/location | Predicted: ${predicted:,.0f}/location')
    
    row = X_test.iloc[pos].values
    print(f'Features: {dict(zip(feature_cols, row))}')
    
    exp = lime_explainer.explain_instance(row, rf.predict, num_features=len(feature_cols))
    
    fig = exp.as_pyplot_figure()
    fig.set_size_inches(8, 4)
    plt.title(f'LIME — {label} (${predicted:,.0f}/loc)')
    plt.tight_layout()
    plt.show()
    
    print('Feature contributions:')
    for feat_rule, weight in exp.as_list():
        print(f'  {weight:+.4f}  {feat_rule}')

In [ ]:
# LIME global feature importance (average over 100 samples)
n_samples = min(100, len(X_test))
sample_idx = np.random.RandomState(42).choice(len(X_test), n_samples, replace=False)

lime_importances = {feat: [] for feat in feature_cols}
for i in sample_idx:
    exp = lime_explainer.explain_instance(X_test.iloc[i].values, rf.predict, num_features=len(feature_cols))
    for feat_rule, weight in exp.as_list():
        matched = False
        for feat in sorted(feature_cols, key=len, reverse=True):
            if feat in feat_rule:
                lime_importances[feat].append(abs(weight))
                matched = True
                break
        if not matched:
            print(f'Unmatched rule: {feat_rule}')

lime_global = pd.Series({f: np.mean(v) if v else 0 for f, v in lime_importances.items()}).sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
lime_global.plot.barh(ax=ax, color='seagreen')
ax.set_xlabel('Mean |LIME weight| (over 100 samples)')
ax.set_title('LIME Global Feature Importance')
plt.tight_layout()
plt.show()

print(lime_global.sort_values(ascending=False).to_string())

In [ ]:
# Side-by-side: RF Gini vs SHAP vs LIME importance
shap_global = pd.Series(np.abs(shap_values).mean(axis=0), index=feature_cols)
rf_global = pd.Series(rf.feature_importances_, index=feature_cols)

comp = pd.DataFrame({
    'RF Gini': rf_global / rf_global.sum(),
    'SHAP': shap_global / shap_global.sum(),
    'LIME': lime_global / lime_global.sum(),
}).sort_values('SHAP', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(comp))
w = 0.25
ax.barh(x - w, comp['RF Gini'], w, label='RF Gini', color='steelblue')
ax.barh(x, comp['SHAP'], w, label='SHAP', color='coral')
ax.barh(x + w, comp['LIME'], w, label='LIME', color='seagreen')
ax.set_yticks(x)
ax.set_yticklabels(comp.index)
ax.set_xlabel('Normalized Importance')
ax.set_title('Feature Importance: RF Gini vs SHAP vs LIME')
ax.legend()
plt.tight_layout()
plt.show()

print(comp.round(3).to_string())